In [ ]:
import pandas as pd
import numpy as np
import gdown
from scipy.sparse import csr_matrix
import time

**1. Pobieranie danych**

In [ ]:
ID_TRAIN = "1zjZe-u52WdE4obfOj8c1foCysYuJvfjB"
ID_TEST = "1y2AMXgEhL3NCT-0Z0nydxSJ6hI_wnrYK"

print("Pobieranie zbioru treningowego:")
gdown.download(f"https://drive.google.com/uc?id={ID_TRAIN}", "df_train.csv", quiet=False)

print("Pobieranie zbioru testowego:")
gdown.download(f"https://drive.google.com/uc?id={ID_TEST}", "df_test.csv", quiet=False)

# Wczytanie zbiorów
df_train = pd.read_csv("df_train.csv")
df_test = pd.read_csv("df_test.csv")

print(f"Dane treningowe: {len(df_train)} wierszy, dane testowe: {len(df_test)} wierszy")

Pobieranie zbioru treningowego:


Downloading...
From: https://drive.google.com/uc?id=1zjZe-u52WdE4obfOj8c1foCysYuJvfjB
To: /content/df_train.csv
100%|██████████| 64.5M/64.5M [00:00<00:00, 110MB/s]


Pobieranie zbioru testowego:


Downloading...
From: https://drive.google.com/uc?id=1y2AMXgEhL3NCT-0Z0nydxSJ6hI_wnrYK
To: /content/df_test.csv
100%|██████████| 11.7M/11.7M [00:00<00:00, 49.8MB/s]


Dane treningowe: 2076316 wierszy, dane testowe: 377262 wierszy


Jak wyglądają dane?

In [ ]:
df_train.head()

,user,item,event,timestamp
0,1001518,94832,click,1581465600
1,2062683,52240,click,1581465600
2,2726272,135586,click,1581465600
3,1666865,4593,click,1581465601
4,129722,69676,click,1581465603


**2. Tworzenie słowników i par bezpośrednich przejść (item_t -> item_{t+1}) dla każdego użytkownika**

Tworzymy listę unikalnych produktów (item) występujących w zbiorze treningowym. Każdemu produktowi przypisujemy unikalny indeks. Analogicznie dla użytkowników.

In [ ]:
unique_items = df_train['item'].unique()
item2idx = {item: idx for idx, item in enumerate(unique_items)}
idx2item = {idx: item for item, idx in item2idx.items()}
n_items = len(unique_items)

unique_users = df_train['user'].unique()
user2idx = {user: idx for idx, user in enumerate(unique_users)}
idx2user = {idx: user for user, idx in user2idx.items()}
n_users = len(unique_users)

unique_users_test = df_test['user'].unique()
user2idx_test = {user: idx for idx, user in enumerate(unique_users_test)}
idx2user_test = {idx: user for user, idx in user2idx_test.items()}
n_users_test = len(unique_users_test)

print(f"Liczba produktów: {n_items}")
print(f"Liczba użytkowników treningowych: {n_users}")
print(f"Liczba użytkowników testowych: {n_users_test}")

train_sorted = df_train.sort_values(by=['user', 'timestamp'])
train_sorted['next_item'] = train_sorted.groupby('user')['item'].shift(-1)
transitions = train_sorted.dropna(subset=['next_item'])

Liczba produktów: 13317
Liczba użytkowników treningowych: 210631
Liczba użytkowników testowych: 72937


Interakcje sortujemy najpierw według użytkownika, a następnie według znacznika czasu (timestamp). Dzięki temu możemy odtworzyć kolejność kliknięć każdego użytkownika.
item_t -> item_{t+1}

Kolumna next_item wskazuje kolejny kliknięty produkt dla danego użytkownika.
Ostatnia interakcja każdego użytkownika nie ma następnego produktu, dlatego zostaje pominięta przy tworzeniu przejść.

In [ ]:
print(transitions.head())

        user    item  event   timestamp  next_item
839633     4    7525  click  1581823094    65386.0
840230     4   65386  click  1581823257   169493.0
840770     4  169493  click  1581823401   115778.0
840837     4  115778  click  1581823419    23751.0
841195     4   23751  click  1581823517   181168.0


Każde przejście zapisujemy za pomocą indeksów produktów. Będą to indeksy macierzy przejścia.

In [ ]:
row_indices = transitions['item'].map(item2idx).values
col_indices = transitions['next_item'].map(item2idx).values

**3. Tworzenie macierzy zliczeń przejść i normalizacja wierszy do prawdopodobieństw**

Tworzymy macierz przejścia. Każde zaobserwowane przejście zwiększa wartość odpowiedniej komórki macierzy.

In [ ]:
counts_matrix = csr_matrix(
    (np.ones(len(row_indices)), (row_indices, col_indices)),
    shape=(n_items, n_items),
    dtype=np.float32
)

row_sums = np.array(counts_matrix.sum(axis=1)).flatten()
row_sums[row_sums == 0] = 1.0
transition_matrix = counts_matrix.multiply(1.0 / row_sums[:, np.newaxis]).tocsr()

Chcemy określić prawdopodobieństwo przejścia z jednego produktu do drugiego. Sumujemy wartości w każdym wierszu, a następnie dzielimy każdą wartość przez sumę danego wiersza.

**4. Ostatnia kliknięta oferta z train dla danego użytkownika i historia**

In [ ]:
last_train_items = train_sorted.groupby('user')['item'].last().to_dict()

user_train_history = df_train.groupby('user')['item'].apply(set).to_dict()

Ostatni produkt jest potrzebny do wyznaczenia kolejnych rekomendacji zgodnie z modelem Markowa. Historia użytkownika może natomiast służyć do wykluczania produktów, które użytkownik już kliknął.

Globalna lista najpopularniejszych ofert

In [ ]:
popular_items_train = df_train['item'].value_counts().index.tolist()

**5. Generowanie rekomendacji**

In [ ]:
def recommend_mcrs(user, k=10, filter_seen=True):
    last_item = last_train_items.get(user)
    seen_items = user_train_history.get(user, set()) if filter_seen else set()

    recs = []
    if last_item in item2idx:
        item_idx = item2idx[last_item]
        probs = transition_matrix[item_idx].toarray().flatten()

        if probs.sum() > 0:
            top_indices = np.argsort(probs)[::-1]
            for idx in top_indices:
                candidate = idx2item[idx]
                if candidate not in seen_items:
                    recs.append(candidate)
                if len(recs) == k:
                    return recs

    #dopełnienie do k najpopularniejszymi ofertami z treningowych
    for item in popular_items_train:
        if item not in seen_items and item not in recs:
            recs.append(item)
        if len(recs) == k:
            break
    return recs

Jeśli model nie potrafi wskazać wystarczającej liczby produktów, na przykład dlatego, że dla ostatniego produktu użytkownika nie zaobserwowano żadnych przejść, brakujące pozycje uzupełniamy najpopularniejszymi ofertami.

**6. Ewaluacja (precission + recall)**

BASELINE:

In [ ]:
def recommend_baseline(user, k=10, filter_seen=True):
    seen_items = user_train_history.get(user, set()) if filter_seen else set()
    recs = []
    for item in popular_items_train:
        if item not in seen_items:
            recs.append(item)
        if len(recs) == k:
            break
    return recs

Użytkownicy w zbiorach:

In [ ]:
train_users = set(df_train['user'].unique())
test_users = set(df_test['user'].unique())
eval_users = list(train_users.intersection(test_users))

In [ ]:
test_true_inter = df_test.groupby('user')['item'].apply(set).to_dict()

def evaluate(model_func, users, k=10):
    precisions = []
    recalls = []

    for user in users:
        actual = test_true_inter[user]
        if not actual:
            continue

        predicted = model_func(user, k=k)

        hits = len(set(predicted).intersection(actual))
        precisions.append(hits / k)
        recalls.append(hits / len(actual))

    return np.mean(precisions), np.mean(recalls)

Ewaluacja dla Łańcuchów Markova i Baseline

In [ ]:
p_base, r_base = evaluate(recommend_baseline, eval_users, k=10)
print(f"--- BASELINE (Popularity) ---")
print(f"Precision@10: {p_base:.4f}")
print(f"Recall@10:    {r_base:.4f}")

p_mcrs, r_mcrs = evaluate(recommend_mcrs, eval_users, k=10)
print(f"\n--- MCRS (Markov Chain) ---")
print(f"Precision@10: {p_mcrs:.4f}")
print(f"Recall@10:    {r_mcrs:.4f}")

--- BASELINE (Popularity) ---
Precision@10: 0.0014
Recall@10:    0.0025

--- MCRS (Markov Chain) ---
Precision@10: 0.0189
Recall@10:    0.0556


In [ ]:
start = time.perf_counter()

for user in eval_users:
    recommend_mcrs(user, k=10, filter_seen=False)

markov_prediction_time = time.perf_counter() - start

start = time.perf_counter()

for user in eval_users:
    recommend_baseline(user, k=10)

baseline_prediction_time = time.perf_counter() - start

print(f"Markov czas rekomendacji: {markov_prediction_time:.4f} s")
print(f"Baseline czas rekomendacji: {baseline_prediction_time:.4f} s")

print(f"Markov średnio na użytkownika: "
      f"{markov_prediction_time / len(eval_users):.6f} s")

print(f"Baseline średnio na użytkownika: "
      f"{baseline_prediction_time / len(eval_users):.6f} s")

Markov czas rekomendacji: 38.0296 s
Baseline czas rekomendacji: 0.1128 s
Markov średnio na użytkownika: 0.000858 s
Baseline średnio na użytkownika: 0.000003 s


**7. Bez wykluczenia wcześniejszych produktów**

In [ ]:
def recommend_baseline_new(user, k=10, filter_seen=True):
    recs = []
    for item in popular_items_train:
      recs.append(item)
      if len(recs) == k:
          break
    return recs

In [ ]:
def recommend_mcrs_new(user, k=10, filter_seen=False):
    last_item = last_train_items.get(user)

    recs = []
    if last_item in item2idx:
        item_idx = item2idx[last_item]
        probs = transition_matrix[item_idx].toarray().flatten()

        if probs.sum() > 0:
            top_indices = np.argsort(probs)[::-1]
            for idx in top_indices:
                candidate = idx2item[idx]
                recs.append(candidate)
                if len(recs) == k:
                    return recs

        for item in popular_items_train:
          if item in popular_items_train:
            if item not in recs:
              recs.append(item)
          if len(recs) == k:
            break
    return recs

In [ ]:
p_base, r_base = evaluate(recommend_baseline_new, eval_users, k=10)
print(f"--- BASELINE (Popularity) ---")
print(f"Precision@10: {p_base:.4f}")
print(f"Recall@10:    {r_base:.4f}")

p_mcrs_new, r_mcrs_new = evaluate(recommend_mcrs_new, eval_users, k=10)
print(f"\n--- MCRS ze wszystkimi produktami ---")
print(f"Precision@10: {p_mcrs_new:.4f}")
print(f"Recall@10:    {r_mcrs_new:.4f}")

--- BASELINE (Popularity) ---
Precision@10: 0.0019
Recall@10:    0.0032

--- MCRS ze wszystkimi produktami ---
Precision@10: 0.0415
Recall@10:    0.1322


In [ ]:
p_no_filter, r_no_filter = evaluate(
    recommend_mcrs_new, eval_users, k=10
)

p_filter, r_filter = evaluate(
    lambda user, k: recommend_mcrs(
        user, k=k, filter_seen=True
    ),
    eval_users,
    k=10
)

print("---BEZ FILTROWANIA---")
print(f"Precision@10: {p_no_filter:.4f}")
print(f"Recall@10:    {r_no_filter:.4f}")

print("\n---Z FILTROWANIEM---")
print(f"Precision@10: {p_filter:.4f}")
print(f"Recall@10:    {r_filter:.4f}")

---BEZ FILTROWANIA---
Precision@10: 0.0415
Recall@10:    0.1322

---Z FILTROWANIEM---
Precision@10: 0.0189
Recall@10:    0.0556
